# Quantization


In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np

In [2]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

In [3]:
X_train = X_train / 255
X_test = X_test / 255

In [4]:
X_train_flattened = X_train.reshape(len(X_train), 28 * 28)
X_test_flattened = X_test.reshape(len(X_test), 28 * 28)

In [5]:
model = keras.Sequential(
    [
        keras.layers.Flatten(input_shape=(28, 28)),
        keras.layers.Dense(100, activation="relu"),
        keras.layers.Dense(10, activation="sigmoid"),
    ]
)

model.compile(
    optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)

model.fit(X_train, y_train, epochs=5)



Epoch 1/5


1875/1875 [==============================] - 5s 2ms/step - loss: 0.2663 - accuracy: 0.9243
Epoch 2/5
1875/1875 [==============================] - 5s 2ms/step - loss: 0.1201 - accuracy: 0.9652
Epoch 3/5
1875/1875 [==============================] - 4s 2ms/step - loss: 0.0838 - accuracy: 0.9746
Epoch 4/5
1875/1875 [==============================] - 4s 2ms/step - loss: 0.0656 - accuracy: 0.9802
Epoch 5/5
1875/1875 [==============================] - 4s 2ms/step - loss: 0.0507 - accuracy: 0.9845


In [6]:
model.evaluate(X_test, y_test)

313/313 [==============================] - 1s 2ms/step - loss: 0.0802 - accuracy: 0.9737


[0.08019882440567017, 0.9736999869346619]

In [7]:
# model.save("./saved_model/")
model.save("saved_model.keras")

In [8]:
model.save("./saved_model/model.keras")

In [9]:
model.export("./saved_model/1/")

INFO:tensorflow:Assets written to: ./saved_model/1/assets


INFO:tensorflow:Assets written to: ./saved_model/1/assets


Saved artifact at './saved_model/1/'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28), dtype=tf.float32, name='flatten_input')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  1501085863376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1501085863952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1501085863184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1501085864336: TensorSpec(shape=(), dtype=tf.resource, name=None)


# (1) Post training quantization

Without quantization


In [10]:
converter = tf.lite.TFLiteConverter.from_saved_model("./saved_model/1/")
tflite_model = converter.convert()

In [11]:
len(tflite_model)

320016

In [12]:
converter = tf.lite.TFLiteConverter.from_saved_model("./saved_model/1/")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()

In [13]:
len(tflite_quant_model)

86056

In [14]:
with open("./saved_model/tflite_model.tflite", "wb") as f:
    f.write(tflite_model)

with open("./saved_model/tflite_quant_model.tflite", "wb") as f:
    f.write(tflite_quant_model)

# (2) Quantization aware training


pip install tensorflow-model-optimization


In [15]:
import tensorflow_model_optimization as tfmot

In [16]:
quantize_model = tfmot.quantization.keras.quantize_model
q_aware_model = quantize_model(model)
q_aware_model.compile(
    optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
q_aware_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 quantize_layer (QuantizeLa  (None, 28, 28)            3         
 yer)                                                            
                                                                 
 quant_flatten (QuantizeWra  (None, 784)               1         
 pperV2)                                                         
                                                                 
 quant_dense (QuantizeWrapp  (None, 100)               78505     
 erV2)                                                           
                                                                 
 quant_dense_1 (QuantizeWra  (None, 10)                1015      
 pperV2)                                                         
                                                                 
Total params: 79524 (310.64 KB)
Trainable params: 79510 

In [17]:
q_aware_model.fit(X_train, y_train, epochs=1)

1875/1875 [==============================] - 7s 3ms/step - loss: 0.0452 - accuracy: 0.9855


In [18]:
converter = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_q_aware_model = converter.convert()

INFO:tensorflow:Assets written to: C:\Users\USER\AppData\Local\Temp\tmpw50pm9jh\assets


INFO:tensorflow:Assets written to: C:\Users\USER\AppData\Local\Temp\tmpw50pm9jh\assets
d:\Python\TensorFlow\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [19]:
with open("q_aware_model.tflite", "wb") as f:
    f.write(tflite_q_aware_model)